# 230968288
Dhanvithraj Shetty


# Experiment 2

In [1]:
import random

class GridEnvironment:
    def __init__(self):
        self.grid = [[random.choice([True, False]) for _ in range(2)] for _ in range(2)]
        self.agent_position = (random.randint(0, 1), random.randint(0, 1))

    def get_percept(self):
        x, y = self.agent_position
        return self.agent_position, self.grid[x][y]

    def move_agent(self, action):
        x, y = self.agent_position
        if action == "Up" and x > 0:
            self.agent_position = (x - 1, y)
        elif action == "Down" and x < 1:
            self.agent_position = (x + 1, y)
        elif action == "Left" and y > 0:
            self.agent_position = (x, y - 1)
        elif action == "Right" and y < 1:
            self.agent_position = (x, y + 1)

    def suck(self):
        x, y = self.agent_position
        if not self.grid[x][y]:
            self.grid[x][y] = True

    def is_clean(self, position):
        x, y = position
        return self.grid[x][y]



In [6]:
class SimpleReflexAgent:
    def __init__(self, environment):
        self.environment = environment
        self.cleaned_rooms = 0
        self.total_moves = 0

    def run(self, steps):
        for _ in range(steps):
            position, is_clean = self.environment.get_percept()
            if not is_clean:
                self.environment.suck()
                self.cleaned_rooms += 1
            else:
                self.random_move()

    def random_move(self):
        actions = ["Up", "Down", "Left", "Right"]
        valid_actions = []
        for action in actions:
            x, y = self.environment.agent_position
            if action == "Up" and x > 0:
                valid_actions.append(action)
            elif action == "Down" and x < 1:
                valid_actions.append(action)
            elif action == "Left" and y > 0:
                valid_actions.append(action)
            elif action == "Right" and y < 1:
                valid_actions.append(action)
        if valid_actions:
            action = random.choice(valid_actions)
            self.environment.move_agent(action)
            self.total_moves +=1 #does not consider suck as a movement 

env = GridEnvironment()
agent = SimpleReflexAgent(env)
steps = 20
agent.run(steps)

print(f"Number of rooms cleaned: {agent.cleaned_rooms}")
print(f"Total number of movements: {agent.total_moves}")


Number of rooms cleaned: 3
Total number of movements: 17


# Experiment 3

In [7]:
class GridEnvironment:
    def __init__(self, size=2):
        self.size = size
        self.grid = {
            (x, y): random.choice(["Dirty", "Clean"])
            for x in range(size)
            for y in range(size)
        }

    def get_status(self, position):
        return self.grid[position]

    def clean(self, position):
        self.grid[position] = "Clean"


In [8]:
class ModelBasedAgent:
    def __init__(self, environment):
        self.env = environment
        self.position = (random.randint(0, 1), random.randint(0, 1))
        self.model = {}
        self.movements = 0

    def perceive(self):
        return self.position, self.env.get_status(self.position)

    def update_model(self, percept):
        position, status = percept
        self.model[position] = status

    def all_rooms_clean(self):
        if len(self.model) < self.env.size ** 2:
            return False
        return all(status == "Clean" for status in self.model.values())

    def valid_moves(self):
        x, y = self.position
        moves = []

        if x > 0:
            moves.append(("Up", (x - 1, y)))
        if x < self.env.size - 1:
            moves.append(("Down", (x + 1, y)))
        if y > 0:
            moves.append(("Left", (x, y - 1)))
        if y < self.env.size - 1:
            moves.append(("Right", (x, y + 1)))

        return moves

    def choose_action(self):
        current_status = self.model[self.position]

        if current_status == "Dirty":
            return "Suck"

        moves = self.valid_moves()

        for action, pos in moves:
            if pos not in self.model or self.model.get(pos) == "Dirty":
                return action

        return random.choice(moves)[0]

    def act(self, action):
        x, y = self.position

        if action == "Suck":
            self.env.clean(self.position)
            self.model[self.position] = "Clean"

        elif action == "Up":
            self.position = (x - 1, y)
            self.movements += 1

        elif action == "Down":
            self.position = (x + 1, y)
            self.movements += 1

        elif action == "Left":
            self.position = (x, y - 1)
            self.movements += 1

        elif action == "Right":
            self.position = (x, y + 1)
            self.movements += 1

    def run(self):
        while True:
            percept = self.perceive()
            self.update_model(percept)

            if self.all_rooms_clean():
                break

            action = self.choose_action()
            self.act(action)


In [9]:
def run_simulation():
    env = GridEnvironment()
    agent = ModelBasedAgent(env)
    agent.run()

    print("Model:", agent.model)
    print("Total movements:", agent.movements)


if __name__ == "__main__":
    run_simulation()

Model: {(0, 1): 'Clean', (1, 1): 'Clean', (1, 0): 'Clean', (0, 0): 'Clean'}
Total movements: 3
